In [3]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os
import webbrowser


In [4]:
df=pd.read_csv('data.csv')
print(df.shape)

(1470, 35)


In [5]:
def check_missing(df):
    results=[] 

In [6]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [7]:
def check_missing(df):
    
    results = []  # empty list — we'll fill this with findings
    
    for col in df.columns:  # go through every column one at a time
        
        missing_count = df[col].isnull().sum()  # count the blanks
        missing_pct = round(missing_count / len(df) * 100, 2)  # turn into percentage
        
        # decide how serious it is
        if missing_pct == 0:
            severity = 'GREEN'
            message = 'No missing values. Column is clean.'
        
        elif missing_pct <= 20:
            severity = 'AMBER'
            message = f'{missing_pct}% of values are missing. Review before using this column.'
        
        else:
            severity = 'RED'
            message = f'CRITICAL — {missing_pct}% missing. Analysis from this column is unreliable.'
        
        # package everything up and add it to our results list
        results.append({
            'column': col,
            'missing_count': missing_count,
            'missing_pct': missing_pct,
            'severity': severity,
            'message': message
        })
    
    return results

In [8]:
missing_results= check_missing(df)
print(missing_results[5])

{'column': 'DistanceFromHome', 'missing_count': 0, 'missing_pct': 0.0, 'severity': 'GREEN', 'message': 'No missing values. Column is clean.'}


In [9]:
for result in missing_results:
    print(result['column'], '—', result['severity'], '—', result['message'])

Age — GREEN — No missing values. Column is clean.
Attrition — GREEN — No missing values. Column is clean.
BusinessTravel — GREEN — No missing values. Column is clean.
DailyRate — GREEN — No missing values. Column is clean.
Department — GREEN — No missing values. Column is clean.
DistanceFromHome — GREEN — No missing values. Column is clean.
Education — GREEN — No missing values. Column is clean.
EducationField — GREEN — No missing values. Column is clean.
EmployeeCount — GREEN — No missing values. Column is clean.
EmployeeNumber — GREEN — No missing values. Column is clean.
EnvironmentSatisfaction — GREEN — No missing values. Column is clean.
Gender — GREEN — No missing values. Column is clean.
HourlyRate — GREEN — No missing values. Column is clean.
JobInvolvement — GREEN — No missing values. Column is clean.
JobLevel — GREEN — No missing values. Column is clean.
JobRole — GREEN — No missing values. Column is clean.
JobSatisfaction — GREEN — No missing values. Column is clean.
Marital

In [10]:
print(df.select_dtypes(include='number').columns.tolist())

['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [11]:
print(len(df.select_dtypes(include='number').columns))

26


In [12]:
print(df.isnull().sum().sum())

0


# TEST CELL — artificially introduced missing values to verify checker works

In [14]:
import numpy as np

df_test = df.copy()  # make a copy — never touch the original
df_test.loc[0:50, 'Age'] = np.nan  # blank out first 50 rows of Age column

test_results = check_missing(df_test)

# find just the Age result
for r in test_results:
    if r['column'] == 'Age':
        print(r)


{'column': 'Age', 'missing_count': 51, 'missing_pct': 3.47, 'severity': 'AMBER', 'message': '3.47% of values are missing. Review before using this column.'}


In [15]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1465    False
1466    False
1467    False
1468    False
1469    False
Length: 1470, dtype: bool

In [16]:
def check_duplicates(df):
    
    dup_count = df.duplicated().sum()  # count duplicate rows
    dup_pct = round(dup_count / len(df) * 100, 2)  # as a percentage
    
    if dup_count == 0:
        severity = 'GREEN'
        message = 'No duplicate rows found. Dataset is unique.'
    
    elif dup_pct <= 5:
        severity = 'AMBER'
        message = f'{dup_count} duplicate rows found ({dup_pct}%). These may skew your averages.'
    
    else:
        severity = 'RED'
        message = f'CRITICAL — {dup_count} duplicates ({dup_pct}%). Remove before any analysis.'
    
    return {
        'dup_count': dup_count,
        'dup_pct': dup_pct,
        'severity': severity,
        'message': message
    }

In [17]:
dup_results=check_duplicates(df)

In [18]:
print(dup_results)

{'dup_count': 0, 'dup_pct': 0.0, 'severity': 'GREEN', 'message': 'No duplicate rows found. Dataset is unique.'}


# TEST CELL — duplicate some rows to verify checker

In [20]:

df_test2 = pd.concat([df, df.iloc[0:80]])  # add first 80 rows again at the bottom

test_dup = check_duplicates(df_test2)
print(test_dup)

{'dup_count': 80, 'dup_pct': 5.16, 'severity': 'RED', 'message': 'CRITICAL — 80 duplicates (5.16%). Remove before any analysis.'}


In [21]:
def check_types(df):
    
    results = []
    
    # only check columns currently stored as text
    for col in df.select_dtypes(include='object').columns:
        
        # test 1 — does this text column actually contain numbers?
        converted_num = pd.to_numeric(df[col], errors='coerce')
        num_success = converted_num.notna().sum()
        
        if num_success > len(df) * 0.8:  # if 80%+ of values converted successfully
            results.append({
                'column': col,
                'issue': 'stored as text but contains numbers',
                'severity': 'AMBER',
                'message': f'"{col}" looks like a number column stored as text. Calculations will fail.'
            })
            continue  # already found an issue, skip date check
        
        # test 2 — does this text column actually contain dates?
        converted_date = pd.to_datetime(df[col], errors='coerce')
        date_success = converted_date.notna().sum()
        
        if date_success > len(df) * 0.8:
            results.append({
                'column': col,
                'issue': 'stored as text but contains dates',
                'severity': 'AMBER',
                'message': f'"{col}" looks like a date column stored as text. Date calculations will fail.'
            })
    
    return results

In [22]:
type_results = check_types(df)

if len(type_results) == 0:
    print('No type issues found')
else:
    for r in type_results:
        print(r['column'], '—', r['message'])

No type issues found


C:\Users\cex\AppData\Local\Temp\ipykernel_15568\516373288.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted_date = pd.to_datetime(df[col], errors='coerce')
C:\Users\cex\AppData\Local\Temp\ipykernel_15568\516373288.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted_date = pd.to_datetime(df[col], errors='coerce')
C:\Users\cex\AppData\Local\Temp\ipykernel_15568\516373288.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted_date = pd.to_datetime(df[col], errors='coerce')
C:\Users\cex\AppData\Local\Temp\ipykernel_15568\516373288.py:22: UserWarn

In [23]:
def check_outliers(df):
    
    results = []
    
    numeric_cols = df.select_dtypes(include='number').columns
    
    for col in numeric_cols:
        
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        # find rows outside the bounds
        outlier_rows = df[(df[col] < lower) | (df[col] > upper)]
        outlier_count = len(outlier_rows)
        row_numbers = list(outlier_rows.index[:10])  # first 10 row numbers only
        
        if outlier_count == 0:
            severity = 'GREEN'
            message = f'No outliers found in "{col}".'
        
        elif outlier_count <= 10:
            severity = 'AMBER'
            message = f'{outlier_count} unusual values found in "{col}". Check rows: {row_numbers}'
        
        else:
            severity = 'RED'
            message = f'{outlier_count} outliers in "{col}". Could indicate data entry errors. Check rows: {row_numbers}'
        
        results.append({
            'column': col,
            'outlier_count': outlier_count,
            'row_numbers': row_numbers,
            'lower_bound': round(lower, 2),
            'upper_bound': round(upper, 2),
            'severity': severity,
            'message': message
        })
    
    return results



In [24]:
outlier_results = check_outliers(df)

# show only columns with actual outliers
for r in outlier_results:
    if r['severity'] != 'GREEN':
        print(r['column'], '—', r['severity'], '—', r['message'])

MonthlyIncome — RED — 114 outliers in "MonthlyIncome". Could indicate data entry errors. Check rows: [25, 29, 45, 62, 105, 106, 112, 119, 123, 147]
NumCompaniesWorked — RED — 52 outliers in "NumCompaniesWorked". Could indicate data entry errors. Check rows: [4, 38, 50, 95, 105, 122, 194, 198, 208, 245]
PerformanceRating — RED — 226 outliers in "PerformanceRating". Could indicate data entry errors. Check rows: [1, 6, 7, 8, 21, 26, 44, 46, 47, 48]
StockOptionLevel — RED — 85 outliers in "StockOptionLevel". Could indicate data entry errors. Check rows: [6, 64, 65, 83, 88, 92, 120, 122, 144, 178]
TotalWorkingYears — RED — 63 outliers in "TotalWorkingYears". Could indicate data entry errors. Check rows: [18, 62, 85, 98, 105, 126, 187, 190, 233, 237]
TrainingTimesLastYear — RED — 238 outliers in "TrainingTimesLastYear". Could indicate data entry errors. Check rows: [0, 10, 16, 20, 23, 26, 31, 32, 33, 41]
YearsAtCompany — RED — 104 outliers in "YearsAtCompany". Could indicate data entry error

In [25]:
def check_types(df):
    
    results = []
    
    # only check columns currently stored as text
    for col in df.select_dtypes(include='object').columns:
        
        # test 1 — does this text column actually contain numbers?
        converted_num = pd.to_numeric(df[col], errors='coerce')
        num_success = converted_num.notna().sum()
        
        if num_success > len(df) * 0.8:  # if 80%+ of values converted successfully
            results.append({
                'column': col,
                'issue': 'stored as text but contains numbers',
                'severity': 'AMBER',
                'message': f'"{col}" looks like a number column stored as text. Calculations will fail.'
            })
            continue  # already found an issue, skip date check
        
        # test 2 — does this text column actually contain dates?
        converted_date = pd.to_datetime(df[col], errors='coerce')
        date_success = converted_date.notna().sum()
        
        if date_success > len(df) * 0.8:
            results.append({
                'column': col,
                'issue': 'stored as text but contains dates',
                'severity': 'AMBER',
                'message': f'"{col}" looks like a date column stored as text. Date calculations will fail.'
            })
    
    return results

In [26]:
def calculate_health_score(missing_results, dup_results, type_results, outlier_results):
    
    score = 100
    
    # missing value penalties
    for r in missing_results:
        if r['severity'] == 'RED':
            score -= 15
        elif r['severity'] == 'AMBER':
            score -= 5
    
    # duplicate penalties
    if dup_results['severity'] == 'RED':
        score -= 15
    elif dup_results['severity'] == 'AMBER':
        score -= 5
    
    # type issue penalties
    for r in type_results:
        if r['severity'] == 'AMBER':
            score -= 5
    
    # outlier penalties
    for r in outlier_results:
        if r['severity'] == 'RED':
            score -= 15
        elif r['severity'] == 'AMBER':
            score -= 5
    
    score = max(0, score)  # never go below 0
    
    if score >= 80:
        label = 'HEALTHY — safe to use for analysis'
    elif score >= 60:
        label = 'NEEDS ATTENTION — fix RED issues first'
    else:
        label = 'POOR QUALITY — do not use until cleaned'
    
    return score, label

In [27]:
score, label = calculate_health_score(
    missing_results,
    dup_results,
    type_results,
    outlier_results
)

print(f'Health Score: {score} / 100')
print(f'Status: {label}')

Health Score: 0 / 100
Status: POOR QUALITY — do not use until cleaned


In [28]:
def calculate_health_score(missing_results, dup_results, type_results, outlier_results):
    
    score = 100
    
    # missing value penalties (unchanged)
    for r in missing_results:
        if r['severity'] == 'RED':
            score -= 15
        elif r['severity'] == 'AMBER':
            score -= 5
    
    # duplicate penalties (unchanged)
    if dup_results['severity'] == 'RED':
        score -= 15
    elif dup_results['severity'] == 'AMBER':
        score -= 5
    
    # type issue penalties (unchanged)
    for r in type_results:
        if r['severity'] == 'AMBER':
            score -= 5
    
    # outlier penalties — now proportional
    for r in outlier_results:
        if r['severity'] == 'GREEN':
            continue
        
        # what % of the dataset are outliers in this column?
        outlier_pct = r['outlier_count'] / 1470 * 100
        
        if outlier_pct < 5:
            score -= 3   # small penalty — minor issue
        elif outlier_pct < 15:
            score -= 7   # medium penalty
        else:
            score -= 12  # large penalty — genuinely concerning
    
    score = max(0, score)
    
    if score >= 80:
        label = 'HEALTHY — safe to use for analysis'
    elif score >= 60:
        label = 'NEEDS ATTENTION — fix RED issues first'
    else:
        label = 'POOR QUALITY — do not use until cleaned'
    
    return score, label

In [29]:
score, label = calculate_health_score(
    missing_results,
    dup_results,
    type_results,
    outlier_results
)

print(f'Health Score: {score} / 100')
print(f'Status: {label}')

Health Score: 36 / 100
Status: POOR QUALITY — do not use until cleaned


DataScope — Interactive Data Quality Inspector

Steps 1-6 complete — all checkers built and tested

IBM HR Dataset: Health Score 36/100 — POOR QUALITY

Step 7 next — HTML dashboard generation

In [31]:
name = "DataScope"
html = f"<h1>Welcome to {name}</h1>"
# result: <h1>Welcome to DataScope</h1>

In [86]:
css_styles = """
<style>
    * { margin: 0; padding: 0; box-sizing: border-box; }
    
    body {
        font-family: 'Segoe UI', Arial, sans-serif;
        background: #0f1117;
        color: #e0e0e0;
        padding: 30px;
    }
    
    .header {
        background: #1a1d2e;
        border: 1px solid #2a2d3e;
        border-radius: 12px;
        padding: 30px;
        margin-bottom: 25px;
    }
    
    .header h1 {
        font-size: 2em;
        color: #7c83fd;
        letter-spacing: 3px;
        margin-bottom: 10px;
    }
    
    .header .subtitle {
        color: #888;
        font-size: 0.95em;
        margin-bottom: 20px;
    }
    
    .stats-row {
        display: flex;
        gap: 20px;
        flex-wrap: wrap;
    }
    
    .stat-box {
        background: #0f1117;
        border: 1px solid #2a2d3e;
        border-radius: 8px;
        padding: 15px 25px;
        text-align: center;
    }
    
    .stat-box .number {
        font-size: 1.8em;
        font-weight: bold;
        color: #7c83fd;
    }
    
    .stat-box .label {
        font-size: 0.8em;
        color: #888;
        margin-top: 4px;
    }
    
    .score-box {
        background: #0f1117;
        border-radius: 8px;
        padding: 15px 25px;
        text-align: center;
    }
    
    .score-box .number { font-size: 1.8em; font-weight: bold; }
    .score-box .label { font-size: 0.8em; color: #888; margin-top: 4px; }
    
    .score-healthy { border: 1px solid #00c896; color: #00c896; }
    .score-attention { border: 1px solid #f0a500; color: #f0a500; }
    .score-poor { border: 1px solid #ff4757; color: #ff4757; }
    
    .section-title {
        font-size: 0.8em;
        letter-spacing: 2px;
        color: #555;
        margin-bottom: 12px;
        text-transform: uppercase;
    }
    
    .columns-grid {
        display: grid;
        grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
        gap: 10px;
        margin-bottom: 25px;
    }
    
    .col-card {
        background: #1a1d2e;
        border-radius: 8px;
        padding: 14px 18px;
        cursor: pointer;
        transition: all 0.2s;
        display: flex;
        align-items: center;
        gap: 12px;
        border: 1px solid #2a2d3e;
    }
    
    .col-card:hover {
        transform: translateY(-2px);
        border-color: #7c83fd;
    }
    
    .dot {
        width: 10px;
        height: 10px;
        border-radius: 50%;
        flex-shrink: 0;
    }
    
    .dot-green { background: #00c896; }
    .dot-amber { background: #f0a500; }
    .dot-red { background: #ff4757; }
    
    .col-name {
        font-size: 0.9em;
        font-weight: 500;
        flex: 1;
    }
    
    .col-badge {
        font-size: 0.7em;
        padding: 2px 8px;
        border-radius: 10px;
        font-weight: bold;
    }
    
    .badge-green { background: #00c89620; color: #00c896; }
    .badge-amber { background: #f0a50020; color: #f0a500; }
    .badge-red { background: #ff475720; color: #ff4757; }
    
    .detail-panel {
        display: none;
        background: #1a1d2e;
        border: 1px solid #2a2d3e;
        border-radius: 12px;
        padding: 25px;
        margin-bottom: 15px;
    }
    
    .detail-panel h3 {
        color: #7c83fd;
        margin-bottom: 5px;
    }
    
    .detail-panel .impact {
        background: #0f1117;
        border-left: 3px solid #7c83fd;
        padding: 12px 16px;
        border-radius: 4px;
        margin: 15px 0;
        font-size: 0.9em;
        color: #aaa;
    }
    
    .row-list {
        font-family: monospace;
        font-size: 0.85em;
        color: #888;
        background: #0f1117;
        padding: 12px;
        border-radius: 6px;
        margin-top: 10px;
    }
    
    .export-btn {
        background: #7c83fd;
        color: white;
        border: none;
        padding: 14px 35px;
        border-radius: 8px;
        font-size: 1em;
        cursor: pointer;
        letter-spacing: 1px;
        margin-top: 10px;
    }
    
    .export-btn:hover { background: #6970e8; }
</style>
"""

In [88]:
javascript = """
<script>
    function showDetail(colId) {
        // hide all panels first
        var panels = document.querySelectorAll('.detail-panel');
        panels.forEach(function(panel) {
            panel.style.display = 'none';
        });
        
        // show the clicked one
        var target = document.getElementById('detail-' + colId);
        if (target) {
            target.style.display = 'block';
            target.scrollIntoView({ behavior: 'smooth', block: 'start' });
        }
    }
</script>
"""

In [90]:
def generate_html_report(df, filename, missing_results, dup_results, 
                          type_results, outlier_results, score, label):
    
    # --- score colour ---
    if score >= 80:
        score_class = 'score-healthy'
    elif score >= 60:
        score_class = 'score-attention'
    else:
        score_class = 'score-poor'
    
    # --- build column cards ---
    # first collect all severities into one easy lookup dictionary
    col_severity = {}
    col_messages = {}
    col_rows = {}
    
    for r in missing_results:
        if r['severity'] != 'GREEN':
            col_severity[r['column']] = r['severity']
            col_messages[r['column']] = r['message']
            col_rows[r['column']] = 'N/A — missing value issue'
    
    for r in outlier_results:
        if r['severity'] != 'GREEN':
            col_severity[r['column']] = r['severity']
            col_messages[r['column']] = r['message']
            col_rows[r['column']] = str(r['row_numbers'])
    
    for r in type_results:
        col_severity[r['column']] = r['severity']
        col_messages[r['column']] = r['message']
        col_rows[r['column']] = 'N/A — type issue affects whole column'
    
    # --- build column cards HTML ---
    cards_html = ''
    details_html = ''
    
    for col in df.columns:
        severity = col_severity.get(col, 'GREEN')
        message = col_messages.get(col, 'No issues found. Column is clean.')
        rows = col_rows.get(col, 'None')
        
        dot_class = f'dot-{severity.lower()}'
        badge_class = f'badge-{severity.lower()}'
        col_id = col.replace(' ', '_')
        
        # the clickable card
        cards_html += f"""
        <div class="col-card" onclick="showDetail('{col_id}')">
            <div class="dot {dot_class}"></div>
            <span class="col-name">{col}</span>
            <span class="col-badge {badge_class}">{severity}</span>
        </div>
        """
        
        # the detail panel (hidden until clicked)
        details_html += f"""
        <div class="detail-panel" id="detail-{col_id}">
            <h3>{col}</h3>
            <p style="color:#888; font-size:0.85em; margin-bottom:10px;">
                Severity: <strong style="color:{'#ff4757' if severity=='RED' else '#f0a500' if severity=='AMBER' else '#00c896'}">{severity}</strong>
            </p>
            <div class="impact">{message}</div>
            <div class="row-list">
                <strong>Affected rows:</strong> {rows}
            </div>
        </div>
        """
    
    # --- duplicate result ---
    dup_html = f"""
    <div class="detail-panel" id="detail-duplicates" style="display:block; margin-bottom:15px;">
        <h3>Duplicate Rows</h3>
        <div class="impact">{dup_results['message']}</div>
    </div>
    """
    
    # --- assemble full HTML ---
    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>DataScope — {filename}</title>
        {css_styles}
    </head>
    <body>
    
    <div class="header">
        <h1>◈ DATASCOPE</h1>
        <p class="subtitle">Interactive Data Quality Inspector</p>
        <div class="stats-row">
            <div class="stat-box">
                <div class="number">{df.shape[0]:,}</div>
                <div class="label">ROWS</div>
            </div>
            <div class="stat-box">
                <div class="number">{df.shape[1]}</div>
                <div class="label">COLUMNS</div>
            </div>
            <div class="stat-box">
                <div class="number">{filename}</div>
                <div class="label">FILE</div>
            </div>
            <div class="score-box {score_class}">
                <div class="number">{score}/100</div>
                <div class="label">{label}</div>
            </div>
        </div>
    </div>
    
    <p class="section-title">Click any column to inspect</p>
    <div class="columns-grid">
        {cards_html}
    </div>
    
    <p class="section-title">Details</p>
    {dup_html}
    {details_html}
    
    <button class="export-btn" onclick="window.print()">
        ⬇ Export Full Report
    </button>
    
    {javascript}
    
    </body>
    </html>
    """
    
    return html

In [92]:
import os

# generate the HTML
html_output = generate_html_report(
    df,
    filename='WA_Fn-UseC_-HR-Employee-Attrition.csv',
    missing_results=missing_results,
    dup_results=dup_results,
    type_results=type_results,
    outlier_results=outlier_results,
    score=score,
    label=label
)

# save it
os.makedirs('../outputs', exist_ok=True)

with open('../outputs/datascope_report.html', 'w', encoding='utf-8') as f:
    f.write(html_output)

print('Report saved!')



Report saved!


True

In [94]:
import os
import webbrowser

# get the full absolute path to the file
abs_path = os.path.abspath('../outputs/datascope_report.html')

print(f'File saved at: {abs_path}')

# open using absolute path
webbrowser.open(f'file:///{abs_path}')

File saved at: C:\Users\cex\outputs\datascope_report.html


True